# Buscador Inteligente de Políticas Internas — Demo ETL + RAG

Este notebook demuestra el pipeline completo:
1. **ETL**: Extracción, limpieza y chunking de documentos
2. **Embeddings**: Conversión de texto a vectores semánticos
3. **ChromaDB**: Almacenamiento vectorial
4. **RAG**: Recuperación + Generación con LLM

**LLM usado**: Groq API (gratuito) — [Obtener clave gratis en console.groq.com](https://console.groq.com)

## PASO 1 — Instalar dependencias

In [ ]:
!pip install -q chromadb sentence-transformers langchain-text-splitters PyMuPDF python-docx groq

## PASO 2 — Configurar API Key de Groq (gratuita)

In [ ]:
import os
from google.colab import userdata

# Opción A: usar Secrets de Colab (recomendado)
# En el panel izquierdo > Secrets > agregar GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except:
    # Opción B: pegar la clave directamente
    GROQ_API_KEY = "gsk_aqui_tu_clave_groq"

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('✓ API Key configurada')

## PASO 3 — Subir documentos (PDF, DOCX o TXT)

In [ ]:
from google.colab import files
import os

os.makedirs('docs', exist_ok=True)
print('Seleccioná uno o más archivos (PDF, DOCX, TXT):')
uploaded = files.upload()

for filename, content in uploaded.items():
    path = os.path.join('docs', filename)
    with open(path, 'wb') as f:
        f.write(content)
    print(f'  ✓ Guardado: {filename} ({len(content):,} bytes)')

## PASO 4 — ETL: Extracción de texto

In [ ]:
from pathlib import Path

def extract_txt(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return [{'text': f.read(), 'page': 1, 'source': Path(path).name}]

def extract_pdf(path):
    import fitz
    pages = []
    doc = fitz.open(path)
    for i, page in enumerate(doc, 1):
        text = page.get_text()
        if not text.strip():
            print(f'  Página {i} escaneada — OCR no disponible en Colab sin Tesseract')
        if text.strip():
            pages.append({'text': text, 'page': i, 'source': Path(path).name})
    return pages

def extract_docx(path):
    from docx import Document
    doc = Document(path)
    text = '\n'.join(p.text for p in doc.paragraphs if p.text.strip())
    return [{'text': text, 'page': 1, 'source': Path(path).name}]

EXTRACTORS = {'.txt': extract_txt, '.pdf': extract_pdf, '.docx': extract_docx}

# Extraer todos los documentos
all_pages = []
for f in Path('docs').iterdir():
    if f.suffix.lower() in EXTRACTORS:
        print(f'Extrayendo: {f.name}')
        pages = EXTRACTORS[f.suffix.lower()](str(f))
        all_pages.extend(pages)
        print(f'  → {len(pages)} página(s) extraída(s)')

print(f'\nTotal: {len(all_pages)} páginas extraídas')

## PASO 5 — ETL: Limpieza y normalización

In [ ]:
import re

def clean(text):
    text = text.replace('\x00', '').replace('\r', '\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^[-=_*]{3,}\s*$', '', text, flags=re.MULTILINE)
    return '\n'.join(line.strip() for line in text.split('\n')).strip()

clean_pages = []
for page in all_pages:
    cleaned = clean(page['text'])
    if len(cleaned) > 50:
        clean_pages.append({**page, 'text': cleaned})

print(f'Páginas antes de limpieza: {len(all_pages)}')
print(f'Páginas después de limpieza: {len(clean_pages)}')
print('\n--- Muestra del texto limpio (primera página) ---')
print(clean_pages[0]['text'][:500] if clean_pages else 'Sin contenido')

## PASO 6 — ETL: Chunking (división en fragmentos)

In [ ]:
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],
    length_function=len,
)

chunks = []
for page in clean_pages:
    parts = splitter.split_text(page['text'])
    for i, part in enumerate(parts):
        if part.strip():
            chunks.append({
                'id': str(uuid.uuid4()),
                'text': part.strip(),
                'source': page['source'],
                'page': page['page'],
                'chunk_index': i,
            })

print(f'Total de chunks generados: {len(chunks)}')
print(f'Tamaño promedio: {sum(len(c["text"]) for c in chunks) // len(chunks)} caracteres')
print()
for src in set(c['source'] for c in chunks):
    n = sum(1 for c in chunks if c['source'] == src)
    print(f'  {src}: {n} chunks')

print('\n--- Ejemplo de chunk ---')
print(chunks[0]['text'])

## PASO 7 — Embeddings + ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

print('Cargando modelo de embeddings (primera vez puede tardar)...')
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
print('✓ Modelo cargado')

# ChromaDB en memoria (Colab es ephemeral)
client = chromadb.Client()
try:
    client.delete_collection('politicas')
except:
    pass
collection = client.create_collection('politicas', metadata={'hnsw:space': 'cosine'})

# Generar embeddings e insertar
print('Generando embeddings e insertando en ChromaDB...')
texts = [c['text'] for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True).tolist()

collection.add(
    ids=[c['id'] for c in chunks],
    documents=texts,
    embeddings=embeddings,
    metadatas=[{'source': c['source'], 'page': c['page']} for c in chunks],
)

print(f'\n✓ {collection.count()} chunks almacenados en ChromaDB')

## PASO 8 — RAG: Función de búsqueda y respuesta

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])

SYSTEM_PROMPT = """Eres un asistente experto en políticas internas de la empresa.
Responde ÚNICAMENTE basándote en el contexto proporcionado.
Si la información no está en los documentos, di: "No encontré información sobre este tema en las políticas internas."
Cita siempre la fuente entre paréntesis. Sé claro y conciso."""

def rag_query(question, top_k=4):
    # 1. Embedding de la pregunta
    q_embedding = embedder.encode([question], normalize_embeddings=True).tolist()[0]

    # 2. Recuperar chunks más similares
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )

    retrieved = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        retrieved.append({'text': doc, 'source': meta['source'], 'page': meta['page'], 'score': round(1 - dist/2, 4)})

    # 3. Construir contexto
    context = '\n\n---\n\n'.join(
        f"[Fuente: {r['source']}, Página {r['page']} | Score: {r['score']}]\n{r['text']}"
        for r in retrieved
    )

    # 4. Generar respuesta con Groq
    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'CONTEXTO:\n{context}\n\nPREGUNTA: {question}'}
        ],
        max_tokens=512,
        temperature=0.1,
    )

    return {
        'answer': response.choices[0].message.content,
        'chunks': retrieved,
        'model': response.model
    }

print('✓ Función RAG lista')

## PASO 9 — Demo interactivo

In [ ]:
# Cambiá esta pregunta por la que quieras probar
pregunta = "¿Cuántos días de vacaciones me corresponden si llevo 3 años en la empresa?"

print(f'Pregunta: {pregunta}')
print('=' * 60)

resultado = rag_query(pregunta)

print('RESPUESTA:')
print(resultado['answer'])
print()
print(f'Modelo usado: {resultado["model"]}')
print(f'Chunks recuperados: {len(resultado["chunks"])}')
print()
print('FRAGMENTOS USADOS:')
for i, c in enumerate(resultado['chunks'], 1):
    print(f'  [{i}] {c["source"]} (pág. {c["page"]}) — Score: {c["score"]}')
    print(f'      {c["text"][:150]}...')
    print()

## PASO 10 — Chat interactivo (múltiples preguntas)

In [ ]:
print('Chat de políticas internas (escribí "salir" para terminar)')
print('=' * 60)

while True:
    try:
        pregunta = input('\nTu pregunta: ').strip()
        if pregunta.lower() in ('salir', 'exit', 'quit', ''):
            print('Cerrando chat.')
            break
        resultado = rag_query(pregunta)
        print(f'\nRespuesta: {resultado["answer"]}')
        fuentes = list({c['source'] for c in resultado['chunks']})
        print(f'Fuentes: {', '.join(fuentes)}')
    except KeyboardInterrupt:
        print('\nChat cerrado.')
        break